# Atualização de Schema — Corpus Jornalístico v2
**Atlas Social de Hortolândia — SMIDS**

## O que este notebook faz

### Em todos os arquivos (diários + compilado):
1. Renomeia `localidade` → `origem`
2. Padroniza `Hortolândia` → `Hortolandia` (sem acento) no campo `origem`
3. Adiciona `municipio_impactado` = `Hortolandia`
4. Adiciona `tipo_abrangencia`: `local` se origem == `Hortolandia`, senão `regional`

### No compilado: também normaliza nomes de colunas

In [1]:
import pandas as pd
import glob
import os
import unicodedata

pasta = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas'
compilado = os.path.join(pasta, 'corpus_resumo_periodos_v07.csv')

arquivos_diarios = sorted(glob.glob(os.path.join(pasta, '????_??_??_tribuna_liberal.csv')))

print(f'Pasta existe: {os.path.exists(pasta)}')
print(f'Compilado existe: {os.path.exists(compilado)}')
print(f'Arquivos diários encontrados: {len(arquivos_diarios)}')

Pasta existe: True
Compilado existe: True
Arquivos diários encontrados: 19


In [2]:
# Função aplicada em todos os arquivos
def adicionar_campos(df):
    # 1. Renomeia localidade → origem
    if 'localidade' in df.columns:
        df = df.rename(columns={'localidade': 'origem'})

    # 2. Remove acento de Hortolândia no campo origem
    if 'origem' in df.columns:
        df['origem'] = df['origem'].astype(str).str.replace('Hortolândia', 'Hortolandia', regex=False)
        df['origem'] = df['origem'].str.replace('Hortolandia', 'Hortolandia', regex=False)

    # 3. Adiciona municipio_impactado após origem
    if 'municipio_impactado' not in df.columns:
        pos = df.columns.get_loc('origem') + 1
        df.insert(pos, 'municipio_impactado', 'Hortolandia')
    else:
        df['municipio_impactado'] = 'Hortolandia'

    # 4. Adiciona tipo_abrangencia após municipio_impactado
    if 'tipo_abrangencia' not in df.columns:
        pos = df.columns.get_loc('municipio_impactado') + 1
        df.insert(pos, 'tipo_abrangencia',
                  df['origem'].apply(
                      lambda x: 'local' if str(x).strip() == 'Hortolandia' else 'regional'
                  ))
    else:
        df['tipo_abrangencia'] = df['origem'].apply(
            lambda x: 'local' if str(x).strip() == 'Hortolandia' else 'regional'
        )

    return df

In [3]:
# Parte 1 — Arquivos diários
total_ok = 0

for caminho in arquivos_diarios:
    nome = os.path.basename(caminho)
    df = pd.read_csv(caminho, encoding='utf-8-sig')
    df = adicionar_campos(df)
    df.to_csv(caminho, index=False, encoding='utf-8')
    total_ok += 1
    print(f'[OK] {nome}')

print('=' * 60)
print(f'Total diários processados: {total_ok}')

[OK] 2026_03_14_tribuna_liberal.csv
[OK] 2026_03_18_tribuna_liberal.csv
[OK] 2026_03_21_tribuna_liberal.csv
[OK] 2026_03_24_tribuna_liberal.csv
[OK] 2026_03_25_tribuna_liberal.csv
[OK] 2026_03_26_tribuna_liberal.csv
[OK] 2026_03_27_tribuna_liberal.csv
[OK] 2026_03_28_tribuna_liberal.csv
[OK] 2026_03_29_tribuna_liberal.csv
[OK] 2026_03_31_tribuna_liberal.csv
[OK] 2026_04_01_tribuna_liberal.csv
[OK] 2026_04_02_tribuna_liberal.csv
[OK] 2026_04_03_tribuna_liberal.csv
[OK] 2026_04_05_tribuna_liberal.csv
[OK] 2026_04_07_tribuna_liberal.csv
[OK] 2026_04_08_tribuna_liberal.csv
[OK] 2026_04_09_tribuna_liberal.csv
[OK] 2026_04_10_tribuna_liberal.csv
[OK] 2026_04_11_tribuna_liberal.csv
Total diários processados: 19


In [4]:
# Parte 2 — Compilado: normaliza nomes de colunas
def normalizar_coluna(nome):
    nome = unicodedata.normalize('NFKD', nome)
    nome = nome.encode('ascii', 'ignore').decode('ascii')
    nome = nome.lower()
    nome = nome.replace(' ', '_').replace('.', '_').replace('/', '_')
    while '__' in nome:
        nome = nome.replace('__', '_')
    nome = nome.rstrip('_')
    return nome

df_comp = pd.read_csv(compilado, encoding='utf-8-sig')

print('Colunas ANTES:')
print(df_comp.columns.tolist())

# Normaliza nomes de colunas
df_comp.columns = [normalizar_coluna(c) for c in df_comp.columns]

print()
print('Colunas APÓS normalização:')
print(df_comp.columns.tolist())

Colunas ANTES:
['data', 'item_evento', 'pagina', 'dimensao_ivs', 'ca3d_variavel', 'tipo_relaaao', 'resumo', 'origem', 'municipio_impactado', 'tipo_abrangencia', 'tipo_evento', 'gravidade', 'observaaao', 'ca3d_lot', 'confianaa', 'polaridade_evento', 'origem\t\t\t\t\t\t']

Colunas APÓS normalização:
['data', 'item_evento', 'pagina', 'dimensao_ivs', 'ca3d_variavel', 'tipo_relaaao', 'resumo', 'origem', 'municipio_impactado', 'tipo_abrangencia', 'tipo_evento', 'gravidade', 'observaaao', 'ca3d_lot', 'confianaa', 'polaridade_evento', 'origem\t\t\t\t\t\t']


In [5]:
# Mapeamento para igualar nomes aos diários
mapa_colunas = {
    'item___evento': 'item',
    'pag': 'pagina',
    'cod_variavel': 'codigo_variavel',
    'tipo_relacao': 'tipo_relacao_variavel',
    'cod_lot': 'cod_loteamento',
    'confianca': 'nivel_confianca_loteamento',
    'polaridade': 'polaridade_evento',
    'origem_1': 'tipo_origem_evento'
}

df_comp = df_comp.rename(columns=mapa_colunas)

print('Colunas APÓS mapeamento:')
print(df_comp.columns.tolist())

Colunas APÓS mapeamento:
['data', 'item_evento', 'pagina', 'dimensao_ivs', 'ca3d_variavel', 'tipo_relaaao', 'resumo', 'origem', 'municipio_impactado', 'tipo_abrangencia', 'tipo_evento', 'gravidade', 'observaaao', 'ca3d_lot', 'confianaa', 'polaridade_evento', 'origem\t\t\t\t\t\t']


In [6]:
# Adiciona os campos novos e salva
df_comp = adicionar_campos(df_comp)
df_comp.to_csv(compilado, index=False, encoding='utf-8')

print(f'Compilado atualizado: {len(df_comp)} linhas')
print()
print('origem únicos:          ', sorted(df_comp['origem'].unique()))
print('municipio_impactado:    ', sorted(df_comp['municipio_impactado'].unique()))
print('tipo_abrangencia únicos:', sorted(df_comp['tipo_abrangencia'].unique()))

Compilado atualizado: 68 linhas

origem únicos:           ['HortolÃ¢ndia', 'Regional']
municipio_impactado:     ['Hortolandia']
tipo_abrangencia únicos: ['regional']


In [7]:
# Verificação final — compara schema diário vs compilado
df_diario = pd.read_csv(arquivos_diarios[0])
df_comp_check = pd.read_csv(compilado)

cols_diario = set(df_diario.columns)
cols_comp = set(df_comp_check.columns)

so_diario = cols_diario - cols_comp
so_comp = cols_comp - cols_diario
iguais = cols_diario & cols_comp

print(f'Colunas iguais ({len(iguais)}): {sorted(iguais)}')
print()
print(f'Só no diário ({len(so_diario)}): {sorted(so_diario)}')
print(f'Só no compilado ({len(so_comp)}): {sorted(so_comp)}')

if not so_diario and not so_comp:
    print()
    print('✓ Schema idêntico em todos os arquivos!')
else:
    print()
    print('⚠ Divergências encontradas — revisar mapeamento.')

Colunas iguais (10): ['data', 'dimensao_ivs', 'gravidade', 'municipio_impactado', 'origem', 'pagina', 'polaridade_evento', 'resumo', 'tipo_abrangencia', 'tipo_evento']

Só no diário (8): ['cod_loteamento', 'codigo_variavel', 'fonte', 'item', 'nivel_confianca_loteamento', 'observacao', 'tipo_origem_evento', 'tipo_relacao_variavel']
Só no compilado (7): ['ca3d_lot', 'ca3d_variavel', 'confianaa', 'item_evento', 'observaaao', 'origem\t\t\t\t\t\t', 'tipo_relaaao']

⚠ Divergências encontradas — revisar mapeamento.
